# WavLM Laughter Detection Training
**Phase A: Frozen encoder + MLP classifier**

Uses WavLM-Base+ for audio-only laugh detection.
Target: F1 >= 0.55 on 50K segments.

In [ ]:
# Setup
!pip install -q transformers librosa soundfile accelerate

import torch
import numpy as np
import json
import time
from collections import Counter
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from sklearn.metrics import f1_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Create sample data (replace with real aligned_segments.jsonl)
samples = []
np.random.seed(42)
for i in range(50000):
    duration = np.random.uniform(0.2, 2.0)
    samples.append({
        "word_index": i,
        "start": i * 0.5,
        "end": i * 0.5 + duration,
        "label": 1 if i % 12 == 0 else 0
    })

with open("aligned_segments.jsonl", "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")

labels = Counter(s["label"] for s in samples)
print(f"Samples: {len(samples)}, laugh={labels[1]} ({100*labels[1]/len(samples):.1f}%)")

In [ ]:
# Load WavLM-Base+
print("Loading WavLM-Base+...")
t0 = time.time()
encoder = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
encoder.to(device)
encoder.eval()
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Model
class WavLMLaughterClassifier(torch.nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.encoder = enc
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 256), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 2)
        )
    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x).last_hidden_state.mean(dim=1)
        return self.classifier(h)

model = WavLMLaughterClassifier(encoder).to(device)
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Train
segments = [json.loads(l) for l in open("aligned_segments.jsonl")]
BATCH, EPOCHS, LR = 64, 3, 1e-3
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([1., 5.]).to(device))

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    preds, labels_list = [], []
    for i in range(0, len(segments), BATCH):
        batch = segments[i:i+BATCH]
        x = torch.randn(len(batch), 16000).to(device)
        y = torch.tensor([s["label"] for s in batch]).to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        preds.extend(model(x).argmax(1).cpu().tolist())
        labels_list.extend(y.cpu().tolist())
    f1 = f1_score(labels_list, preds, zero_division=0)
    print(f"Epoch {ep} ({time.time()-t0:.0f}s) F1={f1:.4f}")

In [ ]:
# Save
torch.save(model.state_dict(), "wavlm_phaseA.pt")
print("Saved: wavlm_phaseA.pt")